# FastAPI JWT Authentication

Demonstrates JWT token creation/verification, password hashing, and a full login flow using FastAPI TestClient.

```bash
pip install fastapi httpx python-jose[cryptography] passlib[bcrypt]
```

In [ ]:
# !pip install fastapi httpx "python-jose[cryptography]" "passlib[bcrypt]"

from jose import jwt, JWTError
from datetime import datetime, timedelta, timezone

# --- JWT configuration ---
SECRET_KEY = "super-secret-key-change-in-production"
ALGORITHM = "HS256"
ACCESS_TOKEN_EXPIRE_MINUTES = 30

def create_access_token(data: dict, expires_delta: timedelta = None) -> str:
    to_encode = data.copy()
    expire = datetime.now(timezone.utc) + (expires_delta or timedelta(minutes=ACCESS_TOKEN_EXPIRE_MINUTES))
    to_encode["exp"] = expire
    return jwt.encode(to_encode, SECRET_KEY, algorithm=ALGORITHM)

def decode_token(token: str) -> dict:
    return jwt.decode(token, SECRET_KEY, algorithms=[ALGORITHM])

# Create a token
token = create_access_token({"sub": "alice", "role": "admin"})
print("Token:", token)

# Decode it
payload = decode_token(token)
print("Payload:", payload)
print("Subject:", payload["sub"])
print("Role:", payload["role"])

In [ ]:
# --- Password hashing with passlib ---
from passlib.context import CryptContext

pwd_context = CryptContext(schemes=["bcrypt"], deprecated="auto")

def hash_password(plain: str) -> str:
    return pwd_context.hash(plain)

def verify_password(plain: str, hashed: str) -> bool:
    return pwd_context.verify(plain, hashed)

password = "mysecretpassword"
hashed = hash_password(password)
print("Plain:  ", password)
print("Hashed: ", hashed)
print("Verify correct:  ", verify_password(password, hashed))
print("Verify wrong:    ", verify_password("wrongpassword", hashed))

In [ ]:
# --- Full login flow with FastAPI + TestClient ---
from fastapi import FastAPI, Depends, HTTPException, status
from fastapi.security import OAuth2PasswordBearer, OAuth2PasswordRequestForm
from fastapi.testclient import TestClient
from pydantic import BaseModel
from jose import JWTError

app = FastAPI()
oauth2_scheme = OAuth2PasswordBearer(tokenUrl="/token")

# Fake user store
USERS_DB = {
    "alice": {"username": "alice", "hashed_password": hash_password("alicepass"), "role": "admin"},
    "bob":   {"username": "bob",   "hashed_password": hash_password("bobpass"),   "role": "user"},
}

def authenticate_user(username: str, password: str):
    user = USERS_DB.get(username)
    if not user or not verify_password(password, user["hashed_password"]):
        return None
    return user

def get_current_user(token: str = Depends(oauth2_scheme)):
    try:
        payload = decode_token(token)
        username = payload.get("sub")
        if not username:
            raise HTTPException(status_code=401, detail="Invalid token")
    except JWTError:
        raise HTTPException(status_code=401, detail="Could not validate token")
    user = USERS_DB.get(username)
    if not user:
        raise HTTPException(status_code=401, detail="User not found")
    return user

@app.post("/token")
def login(form: OAuth2PasswordRequestForm = Depends()):
    user = authenticate_user(form.username, form.password)
    if not user:
        raise HTTPException(status_code=401, detail="Incorrect username or password")
    token = create_access_token({"sub": user["username"], "role": user["role"]})
    return {"access_token": token, "token_type": "bearer"}

@app.get("/me")
def read_me(current_user: dict = Depends(get_current_user)):
    return {"username": current_user["username"], "role": current_user["role"]}

client = TestClient(app)

# Login
r = client.post("/token", data={"username": "alice", "password": "alicepass"})
print("Login:", r.status_code)
token_data = r.json()
access_token = token_data["access_token"]
print("Token type:", token_data["token_type"])

# Access protected endpoint
r = client.get("/me", headers={"Authorization": f"Bearer {access_token}"})
print("/me:", r.json())

# Wrong password
r = client.post("/token", data={"username": "alice", "password": "wrong"})
print("Wrong password:", r.status_code, r.json())

In [ ]:
# --- Token expiry demonstration ---
from jose import ExpiredSignatureError
import time

# Create a token that expires in 1 second
short_token = create_access_token(
    {"sub": "alice"},
    expires_delta=timedelta(seconds=1)
)
print("Short-lived token created.")

# Immediately valid
try:
    payload = decode_token(short_token)
    print("Immediately valid — sub:", payload["sub"])
except JWTError as e:
    print("Error:", e)

# Wait for expiry
print("Waiting 2 seconds for token to expire...")
time.sleep(2)

try:
    payload = decode_token(short_token)
    print("Still valid (unexpected):", payload)
except ExpiredSignatureError:
    print("Token expired — ExpiredSignatureError raised as expected")
except JWTError as e:
    print("JWTError:", e)

In [ ]:
# --- Role-based access control (RBAC) ---
from fastapi import FastAPI, Depends, HTTPException
from fastapi.testclient import TestClient

app2 = FastAPI()
oauth2_scheme2 = OAuth2PasswordBearer(tokenUrl="/token2")

def get_current_user2(token: str = Depends(oauth2_scheme2)):
    try:
        payload = decode_token(token)
        return {"username": payload["sub"], "role": payload.get("role", "user")}
    except JWTError:
        raise HTTPException(status_code=401, detail="Invalid token")

def require_admin(user: dict = Depends(get_current_user2)):
    if user["role"] != "admin":
        raise HTTPException(status_code=403, detail="Admin access required")
    return user

@app2.post("/token2")
def login2(form: OAuth2PasswordRequestForm = Depends()):
    user = authenticate_user(form.username, form.password)
    if not user:
        raise HTTPException(status_code=401, detail="Bad credentials")
    token = create_access_token({"sub": user["username"], "role": user["role"]})
    return {"access_token": token, "token_type": "bearer"}

@app2.get("/admin-only")
def admin_only(user: dict = Depends(require_admin)):
    return {"message": f"Welcome admin {user['username']}"}

client2 = TestClient(app2)

# Admin login
r = client2.post("/token2", data={"username": "alice", "password": "alicepass"})
admin_token = r.json()["access_token"]

# Regular user login
r = client2.post("/token2", data={"username": "bob", "password": "bobpass"})
user_token = r.json()["access_token"]

# Admin accesses admin route
r = client2.get("/admin-only", headers={"Authorization": f"Bearer {admin_token}"})
print("Admin access:", r.json())

# Regular user tries admin route
r = client2.get("/admin-only", headers={"Authorization": f"Bearer {user_token}"})
print("User access:", r.status_code, r.json())